In [1]:
import nmrglue as ng
import numpy as np
import scipy.io as scio


# 将复数数组转换为实数数组，增加一个维度存储实部和虚部
def complex2real(fid_3d):
    '''
    Convert a 2D/3D complex array to a 2D/3D real array 
    with an additional dimension for real and imaginary parts.
    '''
    ndim = fid_3d.ndim
    if ndim == 2:
        N1, N2 = fid_3d.shape
        fid_3d_2c = np.zeros([N1, N2, 2])
        fid_3d_2c[:, :, 0] = np.real(fid_3d)
        fid_3d_2c[:, :, 1] = np.imag(fid_3d)

    elif ndim == 3:
        N1, N2, N3 = fid_3d.shape
        fid_3d_2c = np.zeros([N1, N2, N3, 2])
        fid_3d_2c[:, :, :, 0] = np.real(fid_3d)
        fid_3d_2c[:, :, :, 1] = np.imag(fid_3d)
    return fid_3d_2c

# 将实数数组转换为复数数组
def real2complex(fid_3d_2c):
    ndim = fid_3d_2c.ndim
    if ndim ==3:
        N1, N2, _ = fid_3d_2c.shape
        fid_3d = fid_3d_2c[:, :, 0] + 1j * fid_3d_2c[:, :, 1]
    elif ndim ==4:
        N1, N2, N3, _ = fid_3d_2c.shape
        fid_3d = fid_3d_2c[:, :, :, 0] + 1j * fid_3d_2c[:, :, :, 1]
    return fid_3d

# 将两个3D复数数组转换为超复数格式并保存为NMRPipe的ft1文件
def complex2hyper(recon_r, recon_i, data_type, save_hyper_path):
    '''
    Convert two 3D complex arrays 'Xr' and 'Xi' 
    to a 3D hypercomplex array and save it as a ft1 file.
    '''
    N1,N2,N3 = recon_r.shape
    recon_r_tmp = (recon_r + recon_i)/(2.0)
    recon_i_tmp = (recon_r - recon_i)/(2.0j)
    recon_r = recon_r_tmp
    recon_i = recon_i_tmp
    Rr = np.real(recon_r)
    Ri = np.imag(recon_r)
    Ir = np.real(recon_i)
    Ii = np.imag(recon_i)

    r1 = np.arange(0, N1*2, 2)
    i1 = np.arange(1, N1*2, 2)
    r2 = np.arange(0, N2*2, 2)
    i2 = np.arange(1, N2*2, 2)

    hyper_recon = np.zeros([N1*2,N2*2,N3])
    hyper_recon[np.ix_(r1, r2)] = Rr
    hyper_recon[np.ix_(r1, i2)] = Ri
    hyper_recon[np.ix_(i1, r2)] = Ir
    hyper_recon[np.ix_(i1, i2)] = Ii
    Data = hyper_recon.astype(np.float32)
    print(Data.shape)
    dic, _ = ng.pipe.read(f"./ft1_data/{data_type}_hyper.ft1")  # Note: The specified NUS rate (nus_rate) does not affect the result,
                                                                 # as this operation only loads metadata and spectral data from the file
    
    ng.pipe.write(save_hyper_path, dic, Data, overwrite=True)

def hyper2complex(data_type):
    '''Convert a 3D hypercomplex array (ft1 file, original fid data processed by NMRPipe script proc_dirct.com)
    to two 3D complex arrays Xr and Xi'''
    # _,data = ng.pipe.read(f'./Python code/ft1_data/{data_type}_hyper.ft1')
    _,data = ng.pipe.read(f'F:/NMRPipe/ReadNMRData/{data_type}/{data_type}_hyper.ft1')

    N1 = data.shape[0]
    N2 = data.shape[1]
    N3 = data.shape[2]

    r1 = np.arange(0, N1, 2)
    i1 = np.arange(1, N1, 2)
    r2 = np.arange(0, N2, 2)
    i2 = np.arange(1, N2, 2)

    Rr = data[np.ix_(r1, r2)]
    Ri = data[np.ix_(r1, i2)]
    Ir = data[np.ix_(i1, r2)]
    Ii = data[np.ix_(i1, i2)]
    fid_r = Rr + 1j*Ri #Rc cos(w1*t1)*exp(1i*w2*t2)
    fid_i = Ir + 1j*Ii #Ic sin(w1*t1)*exp(1i*w2*t2)

    fid_1 = fid_r + 1j*fid_i
    fid_2 = fid_r - 1j*fid_i
    return fid_1, fid_2

def complex_nus(data_type,nus_hyper_path):
    '''Perform non-uniform sampling (NUS) on the complex FID data.'''
    fid_r, fid_i = hyper2complex(data_type=data_type)
    fid_r = fid_r/np.max(np.abs(fid_r))  
    fid_i = fid_i/np.max(np.abs(fid_i)) 
    N1_origin,N2_origin,N3 = fid_r.shape
    
    mask = scio.loadmat(f'./NUS_Mask/mask_{data_type}.mat')['mask']
    print('NUS Sampling Rate:', np.sum(mask)/mask.shape[0]/mask.shape[1])
    mask_3d = np.tile(mask[:, :, np.newaxis], (1, 1, N3))

    fid_r_nus = fid_r * mask_3d
    fid_i_nus = fid_i * mask_3d

    fidr_3d_nus_2c = complex2real(fid_r_nus)
    fidi_3d_nus_2c = complex2real(fid_i_nus)
    complex2hyper(real2complex(fidr_3d_nus_2c),real2complex(fidi_3d_nus_2c),data_type,nus_hyper_path)
    print(f'Saved Processed NUS Hypercomplex {data_type} Data')

if __name__ == '__main__':
    data_type = 'Yfgj' # 'Yfgj'|'Altg'|'A3DK08'
    
    # hyper_path = f'./{data_type}_recon.ft1'
    # # recon_r,recon_i分别是对{data_type}_r.mat 与{data_type}_i.mat的欠采重建结果 (N1,N2,N3)
    # recon_r = scio.loadmat('recon_r')['recon_r']
    # recon_i = scio.loadmat('recon_i')['recon_i']
    # recon_r = np.fft.ifft2(recon_r, axes=(0, 1))
    # recon_i = np.fft.ifft2(recon_i, axes=(0, 1))
    # complex2hyper(recon_r,recon_i,data_type,hyper_path)
    
    hyper_path = f'./{data_type}_label.ft1'
    label_r = scio.loadmat('label_r')['label_r']
    label_i = scio.loadmat('label_i')['label_i']
    label_r = np.fft.ifft2(label_r, axes=(0, 1))
    label_i = np.fft.ifft2(label_i, axes=(0, 1))
    complex2hyper(label_r,label_i,data_type,hyper_path)

    # 2.将保存的 './{data_type}_recon.ft1' 在NMRPipe中用proc_indirct.com脚本处理实现填零加窗去掉虚部，
    # 生成的 ./{data_type}_recon.ft结果可以直接沿着直接维投影查看结果


(68, 68, 512)


In [1]:
import numpy as np
import nmrglue as ng
import scipy.io as scio

data_type = 'Yfgj' # 'Yfgj'|'Altg'|'A3DK08'
def peak(data_type):
    if data_type == 'A3DK08':
        peak_position = [327, 331, 339, 344, 345, 352, 353, 358, 359, 361,
                         362, 363, 366, 367, 369, 370, 375, 377, 384, 385,
                         386, 388, 389, 391, 394, 395, 398, 399, 401, 402,
                         403, 405, 407, 409, 412, 415, 417, 419, 422, 424,
                         434]
    elif data_type == 'Altg':
        peak_position = [382, 388, 393, 394, 397, 399, 402, 403, 406, 408,
                         411, 412, 414, 415, 416, 417, 420, 421, 424, 428,
                         429, 430, 432, 433, 435, 437, 439, 441, 442, 443,
                         444, 445, 447, 452, 453, 456, 458, 462]
    elif data_type == 'Yfgj':
        peak_position = [391, 395, 397, 399, 400, 401, 402, 404, 409, 410,
                         411, 412, 413, 415, 416, 418, 419, 421, 422, 424,
                         425, 427, 428, 430, 431, 432, 435, 438, 440, 444,
                         447, 448, 458, 461, 463, 465, 468]
    return peak_position

peak_position = peak(data_type)


dic, recon_spec = ng.pipe.read(f"./{data_type}_recon_spec.ft")
recon_spec = recon_spec.transpose(2, 0, 1)
recon_spec = np.max(recon_spec[:, :, peak_position], 2) / np.max(np.max(recon_spec[:, :, peak_position], 2))
recon_spec = np.fft.fftshift(recon_spec)
# recon_spec = np.max(recon_spec[:, :, iz1-1:izn-1], 2) / np.max(np.max(recon_spec[:, :, iz1-1:izn-1], 2))
# recon_spec = np.max(recon_spec[:, :, :], 2) / np.max(np.max(recon_spec[:, :, :], 2))


dic, label_spec = ng.pipe.read(f"./{data_type}_label_spec.ft")
label_spec = label_spec.transpose(2, 0, 1)
label_spec = np.max(label_spec[:, :, peak_position], 2) / np.max(np.max(label_spec[:, :, peak_position], 2))
label_spec = np.fft.fftshift(label_spec)
# label_spec = np.max(label_spec[:, :, iz1-1:izn-1], 2) / np.max(np.max(label_spec[:, :, iz1-1:izn-1], 2))
# label_spec = np.max(label_spec[:, :, :], 2) / np.max(np.max(label_spec[:, :, :], 2))

print(label_spec.shape)

scio.savemat(f'./{data_type}_recon_spec.mat', {'recon_spec': recon_spec})
scio.savemat(f'./{data_type}_label_spec.mat', {'label_spec': label_spec})

(34, 34)
